# The significance of the existence of salting

> **大数据的 Hash 传送门有一个绝对死板的物理铁律：【Key 相同的数据，必须飞向同一个 Task 小兵的内存】。**
> **当你遇到了恶性数据倾斜，比如官方旗舰店这个超级 Key（`seller_id = 8888`）独自扛了 500 万行订单时，不管你给集群加多少台机器，这 500 万行数据都注定会像滔天巨浪一样，死死砸进 1 号小兵一个人的肚子里，把他当场撑爆。因为在数学上，`Hash(8888)` 算出来的结果永远只有一个。**
> **机器是死板的，但架构师是活的。既然相同的 Key 无法分流，那我们就【往这个 Key 里面人工掺入随机的盐巴（数字后缀）】，把原本顽固不化的 `8888`，强行变成 `8888_0`、`8888_1`、`8888_2`……**
> **在机器眼里，这瞬间变成了 10 个完全不同的新 Key！于是，Hash 传送门被成功蒙蔽，原本会压垮 1 号小兵的 500 万行巨浪，被极其均匀地分流给了 10 个不同机器上的小兵共同抗压。**
> **这就叫加盐法。它是大数据工程中，用“欺骗机器”的数学手段，强行打破 Hash 物理死锁、拯救单点崩溃的终极手术。**

---

## 一、 加盐法的物理全流程：拉慢镜头看“上下半场”

加盐法绝对不是做完一次洗牌就结束了，因为你把 Key 改成了 `8888_0`，虽然并发算得快，但最后的业务报表里并不存在 `8888_0` 这个商家，你必须得把它**还原回去**。因此，加盐法在工业界是一个标准的“两阶段手术”：

### 阶段1：一阶段局部打散聚合（加盐洗牌）

1. **加盐（Salting）**：在原始的 `seller_id` 后面，利用随机数函数，随机拼接上一个固定范围内的数字（比如 0 到 9）。
* *原数据*：500 万行 `8888`。
* *加盐后*：变成了 10 个组（`8888_0` 到 `8888_9`），每组清爽地分流了 50 万行。


2. **局部计算**：全网 10 个小兵并跑，各自在内存里计算自己对应的那个“盐 Key”的总和（Count 或 Sum）。
3. **一阶段战果**：原本要处理 500 万行的大怪兽，在第一阶段结束时，被 10 个小兵在各自家门口**当场打碎并压缩成了只有 10 行中间统计结果**！

### 阶段2：二阶段最终还原聚合（去盐收尾）

1. **去盐（De-salting）**：把第一阶段收拢上来的那 10 行统计结果的后缀（如 `_0` 到 `_9`）**无情地裁剪掉**，让它们全部变回干净的 `8888`。
2. **二次聚合**：这时候，负责接盘最终结果的最终小兵，一抬头发现，飞进自己内存的**不再是 500 万行原始明细，而仅仅是那 10 个已经压缩好的数字**！
3. **最终战果**：小兵在内存里把这 10 个数字啪啪一加，0.0001 秒输出最终答案。全网没有任何一台机器网卡冒烟，没有任何一处内存溢出。

---

## `groupBy` 场景下的加盐法标准实操

我们在 PySpark 里用代码像素级还原这个“加盐 ➔ 局部聚合 ➔ 去盐 ➔ 最终还原”的工业全套动作。


## 🧂 大数据重工业底座：两阶段加盐法（Salting）抗倾斜秘籍

## 1. 物理本质
* **核心欺骗**：`Key + 随机后缀` ➔ 强行将单一物理 Task 承受的滔天巨浪，均匀分流给 N 个并行 Task 共同抗压。
* **时空闭环**：【一阶段加盐明细抗浪聚合】➔【去盐还原】➔【二阶段中间指标无痛汇总】。

## 2. 工业界开火红线（何时用？）
* 满足三个铁律：**真海量大表** + **审计确诊恶性倾斜 (Max/Avg > 10倍)** + **两表皆大、无法实施 Broadcast 广播优化**。

```

In [0]:
from pyspark.sql import functions as F


#先引入表
oi = spark.table("bronze_order_items")

print("====== 🔍 Analyzing the data distribution of seller_id ======")

seller_dist = oi.groupby("seller_id").agg(F.count("*").alias("row_count"))

seller_stats = seller_dist.agg(
    F.max("row_count").alias("max"),
    F.min("row_count").alias("min"),
    F.mean("row_count").alias("mean"),
    F.stddev("row_count").alias("stddev")
).collect()[0]

max_s, min_s,mean_s, std_s = seller_stats["max"],seller_stats["min"], seller_stats["mean"], seller_stats["stddev"]

skew_ratio_s = max_s / mean_s if mean_s > 0 else 0

print(f"【最大单 Key 记录数】: {max_s} 行")
print(f"【全局平均每 Key 记录数】: {mean_s:.2f} 行")
print(f"【标准差（数据波动剧烈度）】: {std_s:.2f}")
print(f"【倾斜倍率（最大/平均）】: {skew_ratio_s:.2f} 倍")




In [0]:
print("====== 🔍 Analyzing the data distribution of product_id ======")

product_dist = oi.groupby("product_id").agg(F.count("*").alias("row_count"))

product_stats = product_dist.agg(
    F.max("row_count").alias("max"),
    F.min("row_count").alias("min"),
    F.mean("row_count").alias("mean"),
    F.stddev("row_count").alias("stddev")
).collect()[0]

max_p, min_p,mean_p, std_p = product_stats["max"],product_stats["min"], product_stats["mean"], product_stats["stddev"]

skew_ratio_p = max_p / mean_p if mean_p > 0 else 0

print(f"【最大单 Key 记录数】: {max_p} 行")
print(f"【全局平均每 Key 记录数】: {mean_p:.2f} 行")
print(f"【标准差（数据波动剧烈度）】: {std_p:.2f}")
print(f"【倾斜倍率（最大/平均）】: {skew_ratio_p:.2f} 倍")

In [0]:
print("\n👉 贡献流量最大的 Top 5 超级卖家：")
seller_dist.orderBy(F.col("row_count").desc()).show(5)


print("\n" + "="*60 + "\n")

In [0]:
print("\n👉 贡献流量最大的 Top 5 超级产品：")
product_dist.orderBy(F.col("row_count").desc()).show(5)


print("\n" + "="*60 + "\n")


现在已经确认在olist中，卖家和产品都存在严重的数据倾斜问题。并且已经分别找出存在问题最大的数据。

**下一步：对倾斜的key进行Salting处理**

In [0]:
# ==========================================
# 🛰️ 🚀 黄金调优第一阶段：人工加盐，局部轰鸣
# ==========================================

# tips:
# F.rand()        → 生成 0（包含）~ 1（不包含）之间的随机小数
# F.rand() * 10   → 把范围放大到 0 ~ 10 之间的随机小数
# F.floor(...)    → 向下取整，只保留整数部分

# 每一组seller_id再分为10类
df_salted = oi.withColumn("salted_key",
            F.concat(F.col("seller_id"),F.lit("_"),F.floor(F.rand()*10)))

# 统计每类的记录数
df_stage1_agg = df_salted.groupBy("salted_key").agg(F.count("*").alias("partial_count"))

df_stage1_agg.select("partial_count").show()
print("它的原理是将")

In [0]:
# 对比一下如果不加盐会怎么样

df = oi.groupBy("seller_id").agg(F.count("*").alias("partial_count2"))
df.select("partial_count2").show()


In [0]:
# =====================
# 对比：加盐前 vs 加盐后 的 组数
# =====================

# 1. 不加盐：按 seller_id 分组，总共有多少组？
group_before = oi.groupBy("seller_id").count().count()

# 2. 加盐后：按 salted_key 分组，总共有多少组？
group_after = df_stage1_agg.count()

# 输出对比
print(f"不加盐 总组数：{group_before}")
print(f"加盐后 总组数：{group_after}")

由此可看出，实际上加盐后组数其实更多了。但是没一组的任务量更均衡了，实际上每台机器的压力相似，任务会更快，不用多等1了

In [0]:
# ==========================================
# 🛰️ 🚀 黄金调优第二阶段：卸套去盐，无痛收尾
# ==========================================

df_desalted = df_stage1_agg.withColumn("original_seller_id",
                                       F.split("salted_key", "_")[0])

df_desalted.show()

final_clean_result = df_desalted.groupBy("original_seller_id").agg(
    F.sum("partial_count").alias("total_order_count")
)

final_clean_result.show()
final_clean_result.count()

**Summary:**
通过上面观察可知：如果我们要算每个seller_id卖出多少单，按照一般的情况是直接将seller_id分组，然后直接统计。但这样会造成一个问题，就是有的seller_id会有数据倾斜，存在多等一问题。  

而在这里的加盐策略其实就是，将每个seller_id再次打碎,然后再次shuffle。原本是一号机一个人死算seller_id=1的1亿单，但是通过加盐后，所有机子都帮忙算seller_id=1的1亿单，并存储在新表中。如下,

seller_id = 1 
1_1 共有 3000余万单  
1_2 共有 3000余万单  
1_3 共有 3000余万单 

然后再将加盐后的seller_id还原成原seller_id
这样会形成，如上多个salted_key对应一个original_seller_id。最后直接按original_seller_id分组，聚合相应的salted_key的分别订单量即可。

简短点来说就是，我本来一个人算1亿单，现在我干不完，我分成很多份，我让大家帮我一起算这一亿单，然后把算出来的结果存起来。最后再把结果加起来。

Summary:
Based on the observations above, if we want to calculate the number of orders sold by each seller_id, the standard approach is to group by seller_id and aggregate directly. However, this introduces a critical issue: some seller_ids suffer from severe data skew, leading to a "straggler" problem where many fast tasks end up waiting for one slow task.

The salting strategy here works by artificially breaking apart each seller_id to distribute the shuffle load. Instead of Machine 1 being crushed under 100 million orders for seller_id=1, salting forces all available machines to pitch in and process a fraction of those 100 million orders, storing the intermediate results in a new temporary state. For example:

seller_id = 1
1_1: ~30 million orders

1_2: ~30 million orders

1_3: ~40 million orders

After this parallel processing phase, the salted IDs are stripped of their random suffixes and mapped back to their original seller_id.
This maps multiple salted_keys to a single original_seller_id. Finally, we simply group by original_seller_id again and sum up the pre-aggregated order volumes from each salted_key.

To put it simply: I was originally supposed to calculate 100 million orders all by myself, but the workload was overwhelming. So, I chopped it into multiple smaller pieces, had everyone in the cluster help me process those pieces simultaneously, and saved their partial results. In the final step, we just sum those small results together.
